# Tugas 1 — Milestone 3: Analisis Temporal, Spasial, dan Advanced EDA

**Mata Kuliah:** Analisis Big Data  
**Target Check-in:** Pertemuan 8 (Mulai dikerjakan Pertemuan 6)  
**Nama Mahasiswa:** Rizqullah Atsir Dafa Childyasa Nusa
**NIM:** 202310370311119
**Username GitHub:**  dafanusa

---

## Panduan Pengerjaan Milestone 3:
1. **Sumber Data:** Gunakan dataset yang telah dibersihkan pada Milestone 2 (`02_data_cleaning.ipynb`).
2. **Analisis Temporal / Windowing:** Terapkan minimal satu operasi temporal (Polars `group_by_dynamic()`, `rolling_mean()`, `shift()`, atau DuckDB window function `OVER (...)`).
3. **Visualisasi Interaktif:** Bangun minimal 6 visualisasi interaktif informatif menggunakan Plotly Express / Graph Objects atau Altair.
4. **Perumusan Insight:** Tuliskan minimal 5 temuan atau wawasan analitik mendalam berdasarkan hasil visualisasi data di atas.


## Bagian 1: Inisialisasi dan Memuat Data Bersih

Muat dataset hasil pembersihan dari folder `data/` menggunakan Polars.

In [1]:
# ============================================================
# BAGIAN 1. INISIALISASI DAN MEMUAT DATA BERSIH
# ============================================================

from pathlib import Path
import warnings

import polars as pl
import duckdb
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. KONFIGURASI PATH PROYEK
# ------------------------------------------------------------

PROJECT_DIR = Path.cwd()

# Antisipasi jika notebook dijalankan dari folder notebooks/
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_PROCESSED = PROJECT_DIR / "data" / "processed"

PATH_PARQUET = (
    DATA_PROCESSED / "indonesian_religious_corpus_bersih.parquet"
)

PATH_PROFILING = (
    DATA_PROCESSED / "profiling_duckdb_summarize.csv"
)

# ------------------------------------------------------------
# 2. VALIDASI FILE
# ------------------------------------------------------------

if not PATH_PARQUET.exists():
    raise FileNotFoundError(
        f"Dataset Parquet tidak ditemukan: {PATH_PARQUET}\n"
        "Pastikan notebook dijalankan dari folder proyek "
        "'tugas1-abd-2026' dan notebook 02 sudah dijalankan."
    )

# ------------------------------------------------------------
# 3. MEMUAT DATASET HASIL CLEANING
# ------------------------------------------------------------

df = pl.read_parquet(PATH_PARQUET)

print("=" * 65)
print("ANALISIS BIG DATA - MILESTONE 3")
print("INDONESIAN RELIGIOUS CORPUS")
print("=" * 65)

print(f"\nJumlah baris : {df.height:,}")
print(f"Jumlah kolom : {df.width:,}")
print(f"Ukuran data  : {PATH_PARQUET.stat().st_size / (1024**2):.2f} MB")

print("\nDaftar kolom:")
print(df.columns)

print("\nTipe data:")
print(df.schema)

# ------------------------------------------------------------
# 4. PEMERIKSAAN DATA AWAL
# ------------------------------------------------------------

print("\nLima baris pertama:")
display(df.head(5))

print("\nRingkasan statistik:")
display(df.describe())

# ------------------------------------------------------------
# 5. PEMERIKSAAN DATA PROFILING
# ------------------------------------------------------------

if PATH_PROFILING.exists():
    print("\nRingkasan profiling DuckDB:")
    df_profiling = pl.read_csv(PATH_PROFILING)
    display(df_profiling.head(10))
else:
    print(
        "\nFile ringkasan profiling tidak ditemukan. "
        "Analisis tetap dapat dilanjutkan."
    )

# ------------------------------------------------------------
# 6. IDENTIFIKASI KOLOM BERDASARKAN NAMA
# ------------------------------------------------------------

columns_lower = {
    col.lower(): col
    for col in df.columns
}

def find_column(candidates):
    """
    Mencari kolom berdasarkan beberapa kemungkinan nama.
    Mengembalikan None jika tidak ditemukan.
    """
    for candidate in candidates:
        if candidate.lower() in columns_lower:
            return columns_lower[candidate.lower()]
    return None

TEXT_COL = find_column([
    "text", "content", "teks", "isi", "document",
    "dokumen", "sentence", "kalimat", "clean_text",
    "text_clean", "teks_bersih", "isi_teks"
])

DATE_COL = find_column([
    "date", "tanggal", "created_at", "timestamp",
    "published_at", "published_date", "waktu"
])

CATEGORY_COL = find_column([
    "category", "kategori", "religion", "agama",
    "topic", "topik", "label", "jenis"
])

SOURCE_COL = find_column([
    "source", "sumber", "source_name", "asal"
])

print("\n" + "=" * 65)
print("IDENTIFIKASI KOLOM")
print("=" * 65)

print("Kolom teks     :", TEXT_COL)
print("Kolom tanggal  :", DATE_COL)
print("Kolom kategori :", CATEGORY_COL)
print("Kolom sumber   :", SOURCE_COL)

if TEXT_COL is None:
    print(
        "\nPERHATIAN: Kolom teks belum teridentifikasi otomatis. "
        "Periksa nama kolom di atas dan sesuaikan TEXT_COL."
    )

print("\nBagian inisialisasi selesai.")

ANALISIS BIG DATA - MILESTONE 3
INDONESIAN RELIGIOUS CORPUS

Jumlah baris : 3,012,817
Jumlah kolom : 5
Ukuran data  : 120.88 MB

Daftar kolom:
['Label', 'Denomination', 'Location', 'Sentence_Unit', 'Link']

Tipe data:
Schema({'Label': String, 'Denomination': String, 'Location': String, 'Sentence_Unit': String, 'Link': String})

Lima baris pertama:


Label,Denomination,Location,Sentence_Unit,Link
str,str,str,str,str
"""Catholic""","""Katolisitas""","""National""","""Banyak orang yang bertanya-tan…","""https://katolisitas.org"""
"""Catholic""","""Katolisitas""","""National""","""Sebab ketika berselingkuh, ses…","""https://katolisitas.org"""
"""Catholic""","""Katolisitas""","""National""","""Kalian pengen masuk Surga ga?""","""https://katolisitas.org"""
"""Catholic""","""Katolisitas""","""National""","""Tinggallah bersama kami, ya Tu…","""https://katolisitas.org"""
"""Catholic""","""Katolisitas""","""National""","""Perjalanan dua orang murid ke …","""https://katolisitas.org"""



Ringkasan statistik:


statistic,Label,Denomination,Location,Sentence_Unit,Link
str,str,str,str,str,str
"""count""","""3012817""","""3012817""","""3012817""","""3012817""","""3012817"""
"""null_count""","""0""","""0""","""0""","""0""","""0"""
"""mean""",null,null,null,null,null
"""std""",null,null,null,null,null
"""min""","""Catholic""","""Alkitab SABDA""","""International""","""( (21): 87) Inilah tasbih Nabi…","""http://Www.muslimah.or.id"""
"""25%""",null,null,null,null,null
"""50%""",null,null,null,null,null
"""75%""",null,null,null,null,null
"""max""","""Protestant""","""Yufid""","""Sumatra""","""𝐅𝐫𝐚𝐧𝐬 𝐒𝐢𝐧𝐮𝐫𝐚𝐭Mengucapkan terim…","""https://zakat.ldii.or.id/zakat…"



Ringkasan profiling DuckDB:


column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,f64
"""Label""","""VARCHAR""","""Catholic""","""Protestant""",3,null,null,null,null,null,3013172,0.0
"""Denomination""","""VARCHAR""","""Alkitab SABDA""","""Yufid""",68,null,null,null,null,null,3013172,0.0
"""Location""","""VARCHAR""","""International""","""Sumatra""",10,null,null,null,null,null,3013172,0.0
"""Date""","""VARCHAR""",null,null,0,null,null,null,null,null,3013172,100.0
"""Title""","""VARCHAR""",null,null,0,null,null,null,null,null,3013172,100.0
"""Sentence_Unit""","""VARCHAR""","""( (21): 87) Inilah tasbih Nabi…","""𝐅𝐫𝐚𝐧𝐬 𝐒𝐢𝐧𝐮𝐫𝐚𝐭Mengucapkan terim…",1963349,null,null,null,null,null,3013172,0.0
"""Link""","""VARCHAR""","""http://Www.muslimah.or.id""","""https://zakat.ldii.or.id/zakat…",128940,null,null,null,null,null,3013172,0.0



IDENTIFIKASI KOLOM
Kolom teks     : None
Kolom tanggal  : None
Kolom kategori : Label
Kolom sumber   : None

PERHATIAN: Kolom teks belum teridentifikasi otomatis. Periksa nama kolom di atas dan sesuaikan TEXT_COL.

Bagian inisialisasi selesai.


## Bagian 2: Analisis Temporal dan Window Functions (Materi Pertemuan 6)

Terapkan teknik windowing Polars (`group_by_dynamic`, `rolling_mean`, `shift`, atau DuckDB window functions)
untuk menangkap tren waktu, pola musiman, atau pergerakan akumulatif.

In [ ]:
# Contoh penerapan temporal operations pada Polars:
# 1. Downsampling ke interval tertentu (misal: per jam / per hari)
# df_temporal = df.sort("timestamp").group_by_dynamic("timestamp", every="1d").agg([...])
#
# 2. Rata-rata bergerak (moving average)
# df_rolling = df.with_columns(pl.col("nilai").rolling_mean(window_size=7).alias("ma_7"))


## Bagian 3: Minimal 6 Visualisasi Interaktif (Plotly / Altair)

Bangun minimal 6 visualisasi interaktif yang mencakup:
- Visualisasi 1: Tren deret waktu (Line chart dengan range slider)
- Visualisasi 2: Distribusi variabel utama (Box plot / Histogram / Violin plot)
- Visualisasi 3: Korelasi antar-variabel numerik (Heatmap / Scatter plot sampling)
- Visualisasi 4: Komposisi kategori utama (Bar chart / Treemap)
- Visualisasi 5: Analisis siklus musiman / perbandingan kelompok
- Visualisasi 6: Pola spasial atau korelasi multi-dimensi

In [ ]:
# Visualisasi 1: Tren Waktu Interaktif
# fig1 = px.line(...)
# fig1.show()


In [ ]:
# Visualisasi 2: Distribusi dan Outlier
# fig2 = px.box(...)
# fig2.show()


In [ ]:
# Visualisasi 3: Korelasi Antar-Variabel
# fig3 = px.imshow(...)
# fig3.show()


In [ ]:
# Visualisasi 4: Komposisi Kategori
# fig4 = px.bar(...)
# fig4.show()


In [ ]:
# Visualisasi 5: Siklus Musiman / Pola Temporal Berkala
# fig5 = px.density_heatmap(...)
# fig5.show()


In [ ]:
# Visualisasi 6: Analisis Lanjutan / Spasial
# fig6 = px.scatter_geo(...)
# fig6.show()


## Bagian 4: Rumusan 5 Insight Analitik Utama

Tuliskan minimal 5 temuan atau wawasan penting yang Anda peroleh dari visualisasi di atas:

1. **Insight 1 (Tren Waktu & Musiman):**  
   [Jelaskan temuan terkait tren jangka panjang atau pola berulang yang teridentifikasi]

2. **Insight 2 (Distribusi & Anomali):**  
   [Jelaskan temuan terkait sebaran data, ketimpangan, atau keberadaan pencilan penting]

3. **Insight 3 (Korelasi Kunci):**  
   [Jelaskan hubungan signifikan antar-variabel yang dapat dijadikan dasar keputusan bisnis/kebijakan]

4. **Insight 4 (Perilaku Kategori / Segmen):**  
   [Jelaskan perbedaan karakteristik performa antar kelompok/wilayah/kategori produk]

5. **Insight 5 (Rekomendasi Praktis):**  
   [Jelaskan tindakan nyata atau strategi operasional yang direkomendasikan berdasarkan seluruh temuan di atas]
